# Visualize NER evaluation metrics
This notebook reads the gold `test.txt` and prediction file `test_predictions.txt` (generated by `run_ner.py`) and visualizes precision / recall / F1 per label, plus an overall classification report.

In [ ]:
%pip install -q matplotlib seaborn pandas scikit-learn seqeval

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from seqeval.metrics import classification_report, precision_recall_fscore_support

repo_dir = Path('.').resolve()
data_dir = repo_dir / 'data'
output_dir = repo_dir / 'output'
gold_path = data_dir / 'test.txt'
pred_path = output_dir / 'test_predictions.txt'
print('gold:', gold_path)
print('pred:', pred_path)

def read_sequences(path):
    seqs = []
    cur = []
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if line == '':
                if cur:
                    seqs.append(cur)
                    cur = []
            else:
                parts = line.split()
                # label is last column for gold, last column for preds as well
                cur.append(parts[-1])
        if cur:
            seqs.append(cur)
    return seqs

# Read gold and predicted sequences (if pred file missing, this will raise)
if not gold_path.exists():
    raise FileNotFoundError(f'Gold test file not found: {gold_path}')
if not pred_path.exists():
    raise FileNotFoundError(f'Prediction file not found: {pred_path}')

gold_seqs = read_sequences(gold_path)
pred_seqs = read_sequences(pred_path)

if len(gold_seqs) != len(pred_seqs):
    print('Warning: number of sequences mismatch (gold, pred):', len(gold_seqs), len(pred_seqs))

# Truncate to min length in case of mismatch
n = min(len(gold_seqs), len(pred_seqs))
gold_seqs = gold_seqs[:n]
pred_seqs = pred_seqs[:n]

print('Loaded sequences:', n)


In [ ]:
# Compute per-label precision/recall/f1/support
labels, p, r, f1, support = precision_recall_fscore_support(gold_seqs, pred_seqs, average=None)
df = pd.DataFrame({'label': labels, 'precision': p, 'recall': r, 'f1': f1, 'support': support})
df = df.sort_values('f1', ascending=False).reset_index(drop=True)
df['support'] = df['support'].astype(int)
df.head(50)


In [ ]:
# Plot precision/recall/f1 for each label
plt.figure(figsize=(10, max(4, len(df)*0.25)))
sns.set(style='whitegrid')
df_melt = df.melt(id_vars=['label'], value_vars=['precision', 'recall', 'f1'], var_name='metric', value_name='score')
sns.barplot(data=df_melt, x='score', y='label', hue='metric')
plt.title('Per-label Precision / Recall / F1')
plt.xlabel('Score')
plt.xlim(0,1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Print text classification report
report = classification_report(gold_seqs, pred_seqs, digits=4)
print(report)
